In [1]:
class DCAStrategy:
    """
    Dollar-Cost Averaging strategy.

    The strategy buys a fixed USD amount of BTC when:
    1. DCA is enabled
    2. The BTC price has fallen by the configured percentage
       since the previous DCA purchase
    """

    def __init__(
        self,
        buy_amount_usd=500,
        drop_pct=3.0,
        enabled=True
    ):
        self.buy_amount_usd = float(buy_amount_usd)
        self.drop_pct = float(drop_pct)
        self.enabled = enabled

        # Price at which the previous DCA purchase occurred
        self.last_buy_price = None

    def should_buy(self, current_price):
        """
        Determine whether a DCA purchase should occur.

        Returns
        -------
        dict
            Decision information.
        """

        if not self.enabled:
            return {
                "action": "HOLD",
                "reason": "DCA strategy is disabled"
            }

        current_price = float(current_price)

        # No previous purchase price
        if self.last_buy_price is None:
            return {
                "action": "BUY",
                "amount_usd": self.buy_amount_usd,
                "price": current_price,
                "reason": "Initial DCA purchase"
            }

        price_change_pct = (
            (current_price - self.last_buy_price)
            / self.last_buy_price
        ) * 100

        required_drop = -self.drop_pct

        if price_change_pct <= required_drop:

            return {
                "action": "BUY",
                "amount_usd": self.buy_amount_usd,
                "price": current_price,
                "price_change_pct": price_change_pct,
                "reason": (
                    f"BTC dropped {abs(price_change_pct):.2f}% "
                    f"since last DCA purchase"
                )
            }

        return {
            "action": "HOLD",
            "price": current_price,
            "price_change_pct": price_change_pct,
            "reason": (
                f"BTC has not dropped {self.drop_pct}% "
                "since last DCA purchase"
            )
        }

    def record_buy(self, price):
        """
        Record the price of the most recent DCA purchase.
        """

        self.last_buy_price = float(price)

In [2]:
dca = DCAStrategy(
    buy_amount_usd=500,
    drop_pct=3.0,
    enabled=True
)

In [3]:
decision = dca.should_buy(100000)

print(decision)

{'action': 'BUY', 'amount_usd': 500.0, 'price': 100000.0, 'reason': 'Initial DCA purchase'}


In [4]:
dca.record_buy(100000)

In [5]:
decision = dca.should_buy(99000)

print(decision)

{'action': 'HOLD', 'price': 99000.0, 'price_change_pct': -1.0, 'reason': 'BTC has not dropped 3.0% since last DCA purchase'}


In [6]:
decision = dca.should_buy(97000)

print(decision)

{'action': 'BUY', 'amount_usd': 500.0, 'price': 97000.0, 'price_change_pct': -3.0, 'reason': 'BTC dropped 3.00% since last DCA purchase'}


In [9]:
import pandas as pd
df = pd.read_csv("../../data/btc_indicators.csv")

latest_price = df.iloc[-1]["close"]

print("Current BTC price:", latest_price)

Current BTC price: 77524.59


In [10]:
decision = dca.should_buy(latest_price)

print(decision)

{'action': 'BUY', 'amount_usd': 500.0, 'price': 77524.59, 'price_change_pct': -22.475410000000004, 'reason': 'BTC dropped 22.48% since last DCA purchase'}
